In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml")

print("Path to dataset files:", path)

100%|██████████| 7.61G/7.61G [01:22<00:00, 99.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, to_timestamp, concat_ws
import pyspark.sql.functions as F

In [ ]:
trans_path = path + "/HI-Medium_Trans.csv"
accounts_path = path + "/HI-Medium_accounts.csv"
patterns_path = path + "/HI-Medium_Patterns.txt"

In [ ]:
spark = SparkSession.builder \
    .appName("AML_Preprocessing") \
    .getOrCreate()

trans = spark.read.csv(trans_path, header=True, inferSchema=True)
accounts = spark.read.csv(accounts_path, header=True, inferSchema=True)

In [ ]:
accounts.printSchema()
accounts.show(5)

root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)

+--------------------+-------+--------------+-----------+--------------------+
|           Bank Name|Bank ID|Account Number|  Entity ID|         Entity Name|
+--------------------+-------+--------------+-----------+--------------------+
|     China Bank #561|  53267|     817D00980|2AA1F24F180| Corporation #183669|
|   Spain Bank #18657| 316997|     808BB2280|2AA1EEB8540| Partnership #193780|
|First Bank of Helena| 339367|     8505ED380|2AA206D7790|Sole Proprietorsh...|
|   Mexico Bank #3367|3148419|     8363D4180|2AA2001B1A0| Partnership #133577|
|Switzerland Bank ...|3174937|     842090C80|2AA20224CB0|Sole Proprietorsh...|
+--------------------+-------+--------------+-----------+--------------------+
only showing top 5 rows


In [ ]:
trans.printSchema()
trans.show(5)

root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)

+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|       Timestamp|From Bank| Account2|To Bank| Account4|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|2022/09/01 00:17|       20|800104D70|     20|800104D70|     

# Data Cleaning

In [ ]:
trans.count()

31898238

In [ ]:
trans = trans.withColumn(
    "timestamp",
    to_timestamp("Timestamp","yyyy/MM/dd HH:mm")
)

In [ ]:
trans = trans.withColumn(
    "src",
    concat_ws("_", "From Bank", "Account2")
)

trans = trans.withColumn(
    "dst",
    concat_ws("_", "To Bank", "Account4")
)

In [ ]:
# Normalize currency
exchange_rate = {
    "US Dollar": 1.0,
    "Euro": 1.08,
    "UK Pound": 1.27,
    "Canadian Dollar": 0.75,
    "Australian Dollar": 0.66,
    "Yen": 0.0067,
    "Rupee": 0.012,
    "Mexican Peso": 0.059,
    "Swiss Franc": 1.12,
    "Brazil Real": 0.20,
    "Bitcoin": 30000,
    "Saudi Riyal": 0.27,
    "Shekel": 0.27,
    "Yuan": 0.14,
    "Ruble": 0.011
}

@F.udf('double')
def normalize_amount(amount, currency):
    rate = exchange_rate.get(currency, 1.0) # Default to 1.0 if currency not found
    return amount * rate

trans = trans.withColumn(
    "Amount Paid (USD)",
    normalize_amount(F.col("Amount Paid"), F.col("Payment Currency"))
)

trans = trans.withColumn(
    "Amount Received (USD)",
    normalize_amount(F.col("Amount Received"), F.col("Receiving Currency"))
)

# Create Transaction Graph

node: account

edge: transaction

In [ ]:
edges = trans.select(
    "src",
    "dst",
    "Amount Paid (USD)",
    "timestamp",
    "Payment Format",
    "Is Laundering"
)

#### create account node

In [ ]:

sender = trans.select(col("src").alias("account"))
receiver = trans.select(col("dst").alias("account"))

nodes = sender.union(receiver).distinct()

In [ ]:
# 1. Tăng số lượng partition cho các phép xáo trộn dữ liệu
spark.conf.set("spark.sql.shuffle.partitions", "400")

# 2. Repartition các DataFrame chính
edges = edges.repartition(400)
nodes = nodes.repartition(400)

print("Đã cấu hình lại số partition và thực hiện repartition cho dữ liệu chính.")

Đã cấu hình lại số partition và thực hiện repartition cho dữ liệu chính.


In [ ]:
outgoing = edges.groupBy("src").agg(
    count("*").alias("out_degree"),
    sum("Amount Paid (USD)").alias("total_sent")
)

In [ ]:
incoming = edges.groupBy("dst").agg(
    count("*").alias("in_degree"),
    sum("Amount Paid (USD)").alias("total_received")
)

In [ ]:
features = outgoing.join(
    incoming,
    outgoing.src == incoming.dst,
    "outer"
)

In [ ]:
features.printSchema()
features.show(5)

root
 |-- src: string (nullable = true)
 |-- out_degree: long (nullable = true)
 |-- total_sent: double (nullable = true)
 |-- dst: string (nullable = true)
 |-- in_degree: long (nullable = true)
 |-- total_received: double (nullable = true)

+-----------+----------+-------------------+-----------+---------+------------------+
|        src|out_degree|         total_sent|        dst|in_degree|    total_received|
+-----------+----------+-------------------+-----------+---------+------------------+
|0_8000474C0|       119|          129593.03|0_8000474C0|       32|          17848.86|
|0_800072650|       422|9.161796749039997E7|0_800072650|       63|1365143.9899999995|
|0_800196970|        32|  79095.92000000001|0_800196970|        2|          64882.01|
|0_800232E70|       186|         9404583.76|0_800232E70|       56|37112.999999999985|
|0_8003A9070|         3|           17737.95|0_8003A9070|        2|          16577.91|
+-----------+----------+-------------------+-----------+---------+---

# Parse patterns.txt

In [ ]:
fraud_edges = []
pattern_type = None

with open(patterns_path) as f:
    for line in f:
        line = line.strip()
        if line.startswith("BEGIN"):
            pattern_type = line.split("-")[1].split(":")[0].strip()
        elif line.startswith("END"):
            pattern_type = None
        else:
            parts = line.split(",")
            if len(parts) == 11:
                from_bank = parts[1]
                from_acc = parts[2]
                to_bank = parts[3]
                to_acc = parts[4]

                src = f"{from_bank}_{from_acc}"
                dst = f"{to_bank}_{to_acc}"

                fraud_edges.append((src, dst, pattern_type))

In [ ]:
fraud_edges[:5]

[('00952_8139F54E0', '0111632_8062C56E0', 'STACK'),
 ('0111632_8062C56E0', '008456_81363F620', 'STACK'),
 ('0118693_823D5EB90', '013729_801CF2E60', 'STACK'),
 ('013729_801CF2E60', '0123621_81A7090F0', 'STACK'),
 ('0024750_81363F410', '0213834_808757B00', 'STACK')]

In [ ]:
fraud_df = spark.createDataFrame(
    fraud_edges,
    ["src", "dst", "pattern"]
)

In [ ]:
# account xuất hiện trong pattern
fraud_accounts = fraud_df.select("src").union(
    fraud_df.select("dst")
).distinct()

In [ ]:
# tạo label cho node
nodes = nodes.join(
    fraud_accounts.withColumn("label", F.lit(1)),
    nodes.account == fraud_accounts.src,
    "left"
)

nodes = nodes.fillna({"label": 0})

# Temporal & Behavioral Aggregation

In [ ]:
# transaction count
tx_count = edges.groupBy("src").agg(
    count("*").alias("tx_count")
)

In [ ]:
# unique counterparties
unique_receivers = edges.groupBy("src").agg(
    F.countDistinct("dst").alias("unique_receivers")
)

In [ ]:
# unique senders
unique_senders = edges.groupBy("dst").agg(
    F.countDistinct("src").alias("unique_senders")
)

In [ ]:
# total money flow
sent_amount = edges.groupBy("src").agg(
    sum("Amount Paid (USD)").alias("total_sent")
)

received_amount = edges.groupBy("dst").agg(
    sum("Amount Paid (USD)").alias("total_received")
)

In [ ]:
# average transaction size
avg_tx = edges.groupBy("src").agg(
    F.avg("Amount Paid (USD)").alias("avg_tx_amount")
)

In [ ]:
# transaction frequentcy
edges = edges.withColumn(
    "date",
    F.to_date("timestamp")
)

In [ ]:
daily_tx = edges.groupBy("src", "date").count()

In [ ]:
tx_freq = daily_tx.groupBy("src").agg(
    F.avg("count").alias("avg_tx_per_day")
)

In [ ]:
# time gap between transactions
from pyspark.sql.window import Window

w = Window.partitionBy("src").orderBy("timestamp")
edges = edges.withColumn(
    "prev_time",
    F.lag("timestamp").over(w)
)
edges = edges.withColumn(
    "time_diff",
    F.unix_timestamp("timestamp") - F.unix_timestamp("prev_time")
)

In [ ]:
gap = edges.groupBy("src").agg(
    F.avg("time_diff").alias("avg_time_gap")
)

In [ ]:
# Payment Method Distribution (Pivoted for features)
payment_features = edges.groupBy("src").pivot("Payment Format").count().fillna(0)
# Rename columns to avoid spaces
for col_name in payment_features.columns:
    if col_name != "src":
        payment_features = payment_features.withColumnRenamed(col_name, "pay_" + col_name.replace(" ", "_"))

In [ ]:
# 1. Start with base nodes and rename 'account' to 'src' to match other DFs
final_features = nodes.select(col("account").alias("src"))

# 2. Join features that use 'src' as the key
final_features = final_features.join(tx_count, "src", "left") \
    .join(unique_receivers, "src", "left") \
    .join(sent_amount, "src", "left") \
    .join(avg_tx, "src", "left") \
    .join(tx_freq, "src", "left") \
    .join(gap, "src", "left") \
    .join(payment_features, "src", "left")

# 3. Join features that use 'dst' as the key
final_features = final_features.join(unique_senders, final_features.src == unique_senders.dst, "left")
final_features = final_features.drop(unique_senders.dst)

final_features = final_features.join(received_amount, final_features.src == received_amount.dst, "left")
final_features = final_features.drop(received_amount.dst)

# 4. Fill nulls and join label
final_features = final_features.fillna(0)

# Get labels from original nodes
labels_df = nodes.select(col("account").alias("src"), "label")
final_features = final_features.join(labels_df, "src", "left")

final_features.show(5)

+----------------+--------+----------------+-----------------+------------------+--------------+-----------------+-------+-----------+--------+----------+---------------+----------------+--------+----------------+--------------+----------------+------------------+-----+
|             src|tx_count|unique_receivers|       total_sent|     avg_tx_amount|avg_tx_per_day|     avg_time_gap|pay_ACH|pay_Bitcoin|pay_Cash|pay_Cheque|pay_Credit_Card|pay_Reinvestment|pay_Wire|             dst|unique_senders|             dst|    total_received|label|
+----------------+--------+----------------+-----------------+------------------+--------------+-----------------+-------+-----------+--------+----------+---------------+----------------+--------+----------------+--------------+----------------+------------------+-----+
|     0_800F9BB60|     189|              15|       1566673.95| 8289.280158730158|       11.8125|7298.297872340426|      6|          0|      20|       100|             62|               1|

### Lưu ý:
* **Khi nào nên tăng?** Khi bạn thấy một vài CPU chạy 100% trong khi các CPU khác rảnh rỗi, hoặc dữ liệu trên mỗi partition quá lớn gây tràn bộ nhớ (Out of Memory).
* **Vị trí đặt:** Nên đặt ngay sau khi bạn tạo ra `edges` và `nodes` để tất cả các phép toán `groupBy` và `join` phía sau được hưởng lợi.

In [ ]:
# 1. Standardize Edge Table
edge_table = edges.select(
    col("src"),
    col("dst"),
    col("Amount Paid (USD)").alias("amount"),
    col("timestamp"),
    col("Payment Format").alias("payment_format"),
    col("Is Laundering").alias("is_laundering_tx")
)

print("Edge Table Schema:")
edge_table.printSchema()
edge_table.show(5)

Edge Table Schema:
root
 |-- src: string (nullable = false)
 |-- dst: string (nullable = false)
 |-- amount: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- payment_format: string (nullable = true)
 |-- is_laundering_tx: integer (nullable = true)

+-----------------+-----------------+------------------+-------------------+--------------+----------------+
|              src|              dst|            amount|          timestamp|payment_format|is_laundering_tx|
+-----------------+-----------------+------------------+-------------------+--------------+----------------+
|1140210_83444D9A0|1140210_83444D9A0|          27.63855|2022-09-01 00:20:00|  Reinvestment|               0|
|  78479_81EFFABA0|  78479_81EFFABA0|         430234.36|2022-09-01 00:19:00|  Reinvestment|               0|
| 286282_821CB6F20| 286282_821CB6F20|        7596.14823|2022-09-01 00:18:00|  Reinvestment|               0|
|1109598_82A1F8260| 115803_82D01E640|238.38750000000002|2022-09-01 00:25

In [ ]:
# 2. Standardize Node Table
# Join 'nodes' with original 'accounts' data to get bank_id and entity_id
# Note: original 'accounts' has 'Account Number' and 'Bank ID', we created 'account' as 'BankID_AccNumber'

# Create a key in accounts to join
accounts_with_key = accounts.withColumn("account_id", concat_ws("_", "Bank ID", "Account Number"))

node_table = nodes.select(col("account").alias("account_id"), "label") \
    .join(accounts_with_key.select("account_id", "Bank ID", "Entity ID"), "account_id", "left") \
    .select(
        "account_id",
        col("Bank ID").alias("bank_id"),
        col("Entity ID").alias("entity_id"),
        "label"
    ).distinct()

print("Node Table Schema:")
node_table.printSchema()
node_table.show(5)

Node Table Schema:
root
 |-- account_id: string (nullable = false)
 |-- bank_id: integer (nullable = true)
 |-- entity_id: string (nullable = true)
 |-- label: integer (nullable = false)

+-----------+-------+-----------+-----+
| account_id|bank_id|  entity_id|label|
+-----------+-------+-----------+-----+
|0_8000474C0|      0|2AA1CA625F0|    0|
|0_80006DFE0|      0|2AA1CA626B0|    0|
|0_80006FE20|      0|2AA1CA62830|    0|
|0_800071D00|      0|2AA1CA628F0|    0|
|0_800072650|      0|2AA1CA9B730|    0|
+-----------+-------+-----------+-----+
only showing top 5 rows


In [ ]:
# 3. Standardize Node Features
# Selecting only the requested fields from your existing logic
node_features = final_features.select(
    col("src").alias("account_id"),
    col("tx_count").alias("out_degree"), # Assuming tx_count was based on 'src'
    col("unique_senders"),
    col("unique_receivers"),
    "total_sent",
    "total_received",
    "avg_tx_amount",
    "avg_tx_per_day",
    "avg_time_gap"
)

# Note: To get 'in_degree', we should use the count from incoming transactions
node_features = node_features.join(incoming.select(col("dst").alias("account_id"), col("in_degree")), "account_id", "left").fillna(0)

# Reorder to match your request
node_features = node_features.select(
    "account_id", "in_degree", "out_degree", "unique_senders",
    "unique_receivers", "total_sent", "total_received",
    "avg_tx_amount", "avg_tx_per_day", "avg_time_gap"
)

print("Node Features Schema:")
node_features.printSchema()
node_features.show(5)

Node Features Schema:
root
 |-- account_id: string (nullable = false)
 |-- in_degree: long (nullable = true)
 |-- out_degree: long (nullable = true)
 |-- unique_senders: long (nullable = true)
 |-- unique_receivers: long (nullable = true)
 |-- total_sent: double (nullable = false)
 |-- total_received: double (nullable = false)
 |-- avg_tx_amount: double (nullable = false)
 |-- avg_tx_per_day: double (nullable = false)
 |-- avg_time_gap: double (nullable = false)

+----------------+---------+----------+--------------+----------------+-----------------+------------------+------------------+--------------+-----------------+
|      account_id|in_degree|out_degree|unique_senders|unique_receivers|       total_sent|    total_received|     avg_tx_amount|avg_tx_per_day|     avg_time_gap|
+----------------+---------+----------+--------------+----------------+-----------------+------------------+------------------+--------------+-----------------+
|     0_800F9BB60|       54|       189|          

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Đường dẫn lưu trên Drive (Bạn có thể đổi tên thư mục tùy ý)
output_path = '/content/drive/MyDrive/Mining of Massive Dataset/Processed_Data'

# Lưu dưới dạng Parquet để tối ưu hiệu năng và dung lượng
print("Đang lưu Edge Table...")
edge_table.write.parquet(f"{output_path}/edge_table", mode='overwrite')

print("Đang lưu Node Table...")
node_table.write.parquet(f"{output_path}/node_table", mode='overwrite')

print("Đang lưu Node Features...")
node_features.write.parquet(f"{output_path}/node_features", mode='overwrite')

print(f"Thành công! Dữ liệu đã được lưu tại: {output_path}")

Đang lưu Edge Table...
Đang lưu Node Table...
Đang lưu Node Features...
Thành công! Dữ liệu đã được lưu tại: /content/drive/MyDrive/Mining of Massive Dataset/Processed_Data
